# OntologyQA - Method 1.1 LLM-only Baseline

Notebook này chạy thử **phương pháp 1.1 LLM-only** cho một sample trong `test_questions_v1.0.xlsx` theo chuẩn OOP.

Mục tiêu:
- Đọc sample theo `SAMPLE_ID` trong `.env`.
- Gọi OpenRouter bằng OpenAI SDK.
- Chấm đáp án theo `selected_option == correct_option`.
- Ghi lại `input_tokens`, `output_tokens`, `total_tokens`, `cost`, `round_trip_latency_ms`.
- Lưu `question_type` để sau này thống kê theo subset.

## 1. Cấu hình

Notebook này chạy method 1.1 trên toàn bộ sample hợp lệ trong `test_questions_v1.0.xlsx`.

Kết quả được lưu theo từng run trong `results/method_1_1/<run_id>/` gồm:

- `method_1_1_results.jsonl`: mỗi dòng là một sample, phù hợp để append và debug.
- `method_1_1_results.csv`: bảng phẳng để thống kê accuracy, token, latency, cost theo `question_type`.
- `method_1_1_summary_by_question_type.csv`: thống kê nhanh theo subset `question_type`.

Cell cuối sẽ gọi OpenRouter cho tất cả sample hợp lệ. Nếu chỉ muốn smoke test một sample, đổi `runner.run_all()` thành `runner.run_one(config.sample_id)`.

In [9]:
import csv
import json
import os
import re
import time
import uuid
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

## 2. Domain Models và Config

In [10]:
@dataclass(frozen=True)
class BaselineConfig:
    project_root: Path
    excel_path: Path
    env_path: Path
    openrouter_base_url: str
    model: str
    api_key: str | None
    sample_id: int
    provider_only: str | None = None
    request_delay_seconds: float = 0.0
    method_id: str = "1.1"
    method_name: str = "LLM-only baseline"
    temperature: float = 0
    max_tokens: int = 256

    @property
    def run_id(self) -> str:
        timestamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
        suffix = uuid.uuid4().hex[:8]
        return f"method-1-1-{timestamp}-{suffix}"

    @staticmethod
    def find_project_root(start: Path | None = None) -> Path:
        current = (start or Path.cwd()).resolve()
        for candidate in [current, *current.parents]:
            if (candidate / "test_questions_v1.0.xlsx").exists() and (candidate / ".env").exists():
                return candidate
        raise FileNotFoundError("Không tìm thấy project root chứa test_questions_v1.0.xlsx và .env.")

    @classmethod
    def from_env(cls) -> "BaselineConfig":
        project_root = cls.find_project_root()
        env_path = project_root / ".env"
        load_dotenv(env_path)

        return cls(
            project_root=project_root,
            excel_path=project_root / "test_questions_v1.0.xlsx",
            env_path=env_path,
            openrouter_base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
            model=os.getenv("OPENROUTER_MODEL", "google/gemma-4-26b-a4b-it"),
            api_key=os.getenv("OPENROUTER_API_KEY"),
            sample_id=int(os.getenv("SAMPLE_ID", "1")),
            provider_only=os.getenv("OPENROUTER_PROVIDER_ONLY"),
            request_delay_seconds=float(os.getenv("REQUEST_DELAY_SECONDS", "0")),
        )


@dataclass(frozen=True)
class QuestionSample:
    sample_id: int
    question: str
    question_type: str
    gold_answer: Any
    correct_option: int
    options: list[Any]


@dataclass(frozen=True)
class LLMCallResult:
    content: str
    usage: dict[str, Any]
    round_trip_latency_ms: float
    raw_response: dict[str, Any]


@dataclass(frozen=True)
class EvaluationResult:
    run_id: str
    method_id: str
    method_name: str
    timestamp_utc: str
    sample_id: int
    question_type: str
    question: str
    gold_answer: Any
    correct_option: int
    predicted_option: int | None
    is_correct: bool
    parse_success: bool
    status: str
    error_type: str | None
    error_message: str | None
    model: str
    provider_only: str | None
    temperature: float
    max_tokens: int
    input_tokens: int | None
    output_tokens: int | None
    total_tokens: int | None
    cost: float | None
    round_trip_latency_ms: float | None
    raw_response: str | None

## 3. Dataset Loader

In [11]:
class QuestionDataset:
    def __init__(self, excel_path: Path):
        self.excel_path = excel_path

    @staticmethod
    def _none_if_nan(value: Any) -> Any:
        return None if pd.isna(value) else value

    def _normalize_record(self, record: dict[str, Any]) -> QuestionSample | None:
        question = self._none_if_nan(record.get("vi_question"))
        correct_option = self._none_if_nan(record.get("answer"))
        options = [self._none_if_nan(record.get(f"option_{idx}")) for idx in range(1, 6)]
        available_options = [option for option in options if option is not None]

        if not question or correct_option is None or len(available_options) < 2:
            return None

        return QuestionSample(
            sample_id=int(record["number"]),
            question=str(question),
            question_type=str(self._none_if_nan(record.get("question_type")) or "").strip(),
            gold_answer=self._none_if_nan(record.get("gold_answer")),
            correct_option=int(correct_option),
            options=options,
        )

    def load_valid_samples(self) -> list[QuestionSample]:
        df = pd.read_excel(self.excel_path, engine="openpyxl")
        df = df.rename(columns={"Unnamed: 3": "gold_answer"})

        samples = []
        for record in df.to_dict(orient="records"):
            sample = self._normalize_record(record)
            if sample is not None:
                samples.append(sample)
        return samples

    def get_by_id(self, sample_id: int) -> QuestionSample:
        samples = self.load_valid_samples()
        for sample in samples:
            if sample.sample_id == sample_id:
                return sample

        available_ids = [sample.sample_id for sample in samples]
        raise ValueError(f"Không tìm thấy sample id {sample_id}. Các id hợp lệ: {available_ids}")

## 4. Prompt Builder

In [12]:
class LLMOnlyPromptBuilder:
    SYSTEM_PROMPT = (
        "Bạn là baseline LLM-only cho bài toán OntologyQA. "
        "Không sinh SPARQL, không gọi công cụ, không giả định có truy cập knowledge graph. "
        "Chỉ chọn đúng một đáp án từ các lựa chọn được cung cấp."
    )

    def build_messages(self, sample: QuestionSample) -> list[dict[str, str]]:
        options_text = "\n".join(
            f"{idx}. {option}"
            for idx, option in enumerate(sample.options, start=1)
            if option is not None
        )

        user_prompt = f"""Câu hỏi tiếng Việt:
{sample.question}

Loại câu hỏi: {sample.question_type or 'unknown'}

Các lựa chọn:
{options_text}

Hãy chọn đúng 1 đáp án trong các lựa chọn trên.
Chỉ trả về JSON hợp lệ theo schema:
{{"selected_option": <số nguyên 1-5>, "answer_text": "<nội dung đáp án>", "reason": "<giải thích ngắn>"}}
"""

        return [
            {"role": "system", "content": self.SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]

## 5. OpenRouter Client

In [13]:
class OpenRouterChatClient:
    def __init__(self, config: BaselineConfig):
        if not config.api_key:
            raise RuntimeError("Thiếu OPENROUTER_API_KEY. Hãy điền API key trong .env.")

        self.config = config
        self.client = OpenAI(
            base_url=config.openrouter_base_url,
            api_key=config.api_key,
        )

    def _extra_body(self) -> dict[str, Any] | None:
        if not self.config.provider_only:
            return None

        providers = [provider.strip() for provider in self.config.provider_only.split(",") if provider.strip()]
        return {"provider": {"only": providers}} if providers else None

    def complete(self, messages: list[dict[str, str]]) -> LLMCallResult:
        request_kwargs = {
            "model": self.config.model,
            "messages": messages,
            "temperature": self.config.temperature,
            "max_tokens": self.config.max_tokens,
        }
        extra_body = self._extra_body()
        if extra_body:
            request_kwargs["extra_body"] = extra_body

        started = time.perf_counter()
        completion = self.client.chat.completions.create(**request_kwargs)
        round_trip_latency_ms = (time.perf_counter() - started) * 1000

        raw_response = completion.model_dump()
        content = completion.choices[0].message.content or ""
        usage = raw_response.get("usage") or {}

        return LLMCallResult(
            content=content,
            usage=usage,
            round_trip_latency_ms=round_trip_latency_ms,
            raw_response=raw_response,
        )

## 6. Parser, Evaluator và Logger

In [14]:
class ResponseParser:
    def parse_selected_option(self, text: str) -> int | None:
        cleaned = text.strip()
        fenced = re.search(r"```(?:json)?\s*(.*?)```", cleaned, flags=re.DOTALL | re.IGNORECASE)
        if fenced:
            cleaned = fenced.group(1).strip()

        try:
            payload = json.loads(cleaned)
            selected = payload.get("selected_option")
            return int(selected) if selected is not None else None
        except Exception:
            match = re.search(r"selected_option[^0-9]*(\d+)", cleaned)
            return int(match.group(1)) if match else None


class BaselineEvaluator:
    def __init__(self, config: BaselineConfig):
        self.config = config
        self.parser = ResponseParser()

    def evaluate(self, run_id: str, sample: QuestionSample, llm_result: LLMCallResult) -> EvaluationResult:
        predicted_option = self.parser.parse_selected_option(llm_result.content)
        parse_success = predicted_option is not None
        usage = llm_result.usage

        return EvaluationResult(
            run_id=run_id,
            method_id=self.config.method_id,
            method_name=self.config.method_name,
            timestamp_utc=datetime.now(timezone.utc).isoformat(),
            sample_id=sample.sample_id,
            question_type=sample.question_type,
            question=sample.question,
            gold_answer=sample.gold_answer,
            correct_option=sample.correct_option,
            predicted_option=predicted_option,
            is_correct=predicted_option == sample.correct_option,
            parse_success=parse_success,
            status="success",
            error_type=None,
            error_message=None,
            model=self.config.model,
            provider_only=self.config.provider_only,
            temperature=self.config.temperature,
            max_tokens=self.config.max_tokens,
            input_tokens=usage.get("prompt_tokens"),
            output_tokens=usage.get("completion_tokens"),
            total_tokens=usage.get("total_tokens"),
            cost=usage.get("cost"),
            round_trip_latency_ms=llm_result.round_trip_latency_ms,
            raw_response=llm_result.content,
        )

    def failed(self, run_id: str, sample: QuestionSample, error: Exception, latency_ms: float | None) -> EvaluationResult:
        return EvaluationResult(
            run_id=run_id,
            method_id=self.config.method_id,
            method_name=self.config.method_name,
            timestamp_utc=datetime.now(timezone.utc).isoformat(),
            sample_id=sample.sample_id,
            question_type=sample.question_type,
            question=sample.question,
            gold_answer=sample.gold_answer,
            correct_option=sample.correct_option,
            predicted_option=None,
            is_correct=False,
            parse_success=False,
            status="error",
            error_type=type(error).__name__,
            error_message=str(error),
            model=self.config.model,
            provider_only=self.config.provider_only,
            temperature=self.config.temperature,
            max_tokens=self.config.max_tokens,
            input_tokens=None,
            output_tokens=None,
            total_tokens=None,
            cost=None,
            round_trip_latency_ms=latency_ms,
            raw_response=None,
        )


class ResultLogger:
    def __init__(self, project_root: Path):
        self.project_root = project_root

    def result_dir(self, run_id: str) -> Path:
        return self.project_root / "results" / "method_1_1" / run_id

    def save_all(self, run_id: str, results: list[EvaluationResult]) -> Path:
        output_dir = self.result_dir(run_id)
        output_dir.mkdir(parents=True, exist_ok=True)

        records = [asdict(result) for result in results]
        jsonl_path = output_dir / "method_1_1_results.jsonl"
        with jsonl_path.open("w", encoding="utf-8") as file:
            for record in records:
                file.write(json.dumps(record, ensure_ascii=False) + "\n")

        csv_path = output_dir / "method_1_1_results.csv"
        if records:
            with csv_path.open("w", encoding="utf-8-sig", newline="") as file:
                writer = csv.DictWriter(file, fieldnames=list(records[0].keys()))
                writer.writeheader()
                writer.writerows(records)

            summary = self.summarize(records)
            summary_path = output_dir / "method_1_1_summary_by_question_type.csv"
            summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

        return output_dir

    @staticmethod
    def summarize(records: list[dict[str, Any]]) -> pd.DataFrame:
        df = pd.DataFrame(records)
        successful = df[df["status"] == "success"].copy()
        if successful.empty:
            return pd.DataFrame()

        return (
            successful.groupby("question_type", dropna=False)
            .agg(
                n_samples=("sample_id", "count"),
                accuracy=("is_correct", "mean"),
                parse_success_rate=("parse_success", "mean"),
                avg_input_tokens=("input_tokens", "mean"),
                avg_output_tokens=("output_tokens", "mean"),
                avg_total_tokens=("total_tokens", "mean"),
                total_cost=("cost", "sum"),
                avg_round_trip_latency_ms=("round_trip_latency_ms", "mean"),
            )
            .reset_index()
            .sort_values("question_type")
        )

## 7. Runner OOP cho Method 1.1

In [15]:
class LLMOnlyBaselineRunner:
    def __init__(self, config: BaselineConfig):
        self.config = config
        self.dataset = QuestionDataset(config.excel_path)
        self.prompt_builder = LLMOnlyPromptBuilder()
        self.llm_client = OpenRouterChatClient(config)
        self.evaluator = BaselineEvaluator(config)
        self.logger = ResultLogger(config.project_root)

    def _run_sample(self, run_id: str, sample: QuestionSample) -> EvaluationResult:
        messages = self.prompt_builder.build_messages(sample)
        started = time.perf_counter()
        try:
            llm_result = self.llm_client.complete(messages)
            return self.evaluator.evaluate(run_id, sample, llm_result)
        except Exception as error:
            latency_ms = (time.perf_counter() - started) * 1000
            return self.evaluator.failed(run_id, sample, error, latency_ms)

    def run_one(self, sample_id: int) -> EvaluationResult:
        run_id = self.config.run_id
        sample = self.dataset.get_by_id(sample_id)
        result = self._run_sample(run_id, sample)
        output_dir = self.logger.save_all(run_id, [result])

        print("Sample:")
        print(json.dumps(asdict(sample), ensure_ascii=False, indent=2))
        print("\nResult:")
        print(json.dumps(asdict(result), ensure_ascii=False, indent=2))
        print(f"\nĐã lưu kết quả vào: {output_dir}")

        return result

    def run_all(self) -> list[EvaluationResult]:
        run_id = self.config.run_id
        samples = self.dataset.load_valid_samples()
        results = []

        print(f"Bắt đầu chạy {len(samples)} sample hợp lệ. run_id={run_id}")
        for index, sample in enumerate(samples, start=1):
            print(f"[{index}/{len(samples)}] sample_id={sample.sample_id}, question_type={sample.question_type}")
            result = self._run_sample(run_id, sample)
            results.append(result)
            print(
                f"  status={result.status}, predicted={result.predicted_option}, "
                f"correct={result.correct_option}, is_correct={result.is_correct}, "
                f"tokens={result.total_tokens}, latency_ms={result.round_trip_latency_ms}"
            )

            if self.config.request_delay_seconds > 0 and index < len(samples):
                time.sleep(self.config.request_delay_seconds)

        output_dir = self.logger.save_all(run_id, results)
        successful = [result for result in results if result.status == "success"]
        errors = [result for result in results if result.status == "error"]
        accuracy = sum(result.is_correct for result in successful) / len(successful) if successful else None

        print("\nTổng kết:")
        print(f"  Tổng sample: {len(results)}")
        print(f"  Thành công: {len(successful)}")
        print(f"  Lỗi: {len(errors)}")
        print(f"  Accuracy trên các sample thành công: {accuracy}")
        print(f"  Đã lưu kết quả vào: {output_dir}")

        return results

## 8. Chạy toàn bộ dataset

Cell dưới đây sẽ gọi OpenRouter cho toàn bộ sample hợp lệ trong Excel. Với model free, nếu gặp rate limit 429, dòng sample đó vẫn được ghi với `status="error"`; có thể chạy lại sau để bổ sung hoặc lọc các sample lỗi.

In [16]:
config = BaselineConfig.from_env()
print(f"Project root: {config.project_root}")
print(f"Excel exists: {config.excel_path.exists()} -> {config.excel_path}")
print(f".env exists: {config.env_path.exists()} -> {config.env_path}")
print(f"OpenRouter base URL: {config.openrouter_base_url}")
print(f"Model: {config.model}")
print(f"Provider only: {config.provider_only}")
print(f"Request delay seconds: {config.request_delay_seconds}")
print(f"API key configured: {bool(config.api_key)}")

runner = LLMOnlyBaselineRunner(config)
results = runner.run_all()

Project root: D:\Dev\VDT2026-OntologyQA
Excel exists: True -> D:\Dev\VDT2026-OntologyQA\test_questions_v1.0.xlsx
.env exists: True -> D:\Dev\VDT2026-OntologyQA\.env
OpenRouter base URL: https://openrouter.ai/api/v1
Model: google/gemma-4-26b-a4b-it
Provider only: nextbit
Request delay seconds: 0.0
API key configured: True
Bắt đầu chạy 62 sample hợp lệ. run_id=method-1-1-20260613-163248-dacd0211
[1/62] sample_id=1, question_type=entity
  status=success, predicted=3, correct=3, is_correct=True, tokens=224, latency_ms=1879.7328000000562
[2/62] sample_id=2, question_type=entity
  status=success, predicted=4, correct=4, is_correct=True, tokens=268, latency_ms=1592.8963999976986
[3/62] sample_id=3, question_type=entity
  status=success, predicted=2, correct=5, is_correct=False, tokens=234, latency_ms=1538.4153999984846
[4/62] sample_id=4, question_type=entity
  status=success, predicted=4, correct=4, is_correct=True, tokens=223, latency_ms=2227.635700000974
[5/62] sample_id=5, question_type=e